In [1]:
import os
# Version for filename
#ver = 'GRU'  # GRU or LSTM

date = "251128"
dl_folder = f"D:/Python_TK_3/datas/{date}_DL"

dl_number = "Vis_angle_multi_09"
#os.mkdir(f"{dl_folder}/model_{dl_number}")

In [2]:
# データのサンプリングレート
fs = 20  # サンプリング周波数
first_ex = 0
last_ex = 114
NumberOfDatas = last_ex - first_ex + 1        # number of experiments
start_stim = 20
stop_stim = 30
calc_start = 5
calc_end = 39
start_ave= 5
end_ave= 11
look_frame = 30   

In [3]:
# linear_svm_train_save.py
import os
import json
import joblib
import numpy as np
import pandas as pd
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, roc_auc_score, hinge_loss

def train_linear_svm_and_save(X, y, test_size=0.2, random_state=42,
                              C=1.0, max_iter=20000, tol=1e-3, feature_names=None,
                              use_dual="true", class_weight=None):
    X_tr, X_val, y_tr, y_val = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    """
    X: numpy array or pandas DataFrame of shape (n_samples, n_features)
    y: array-like of shape (n_samples,), with labels 0/1
    feature_names: list of str (optional). If None and X is DataFrame, uses X.columns
    """

    # Prepare output directory (timestamped subdir for reproducibility)
    #os.makedirs(out_dir, exist_ok=True)
    #stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    #out_dir = os.path.join(out_dir, stamp)
    #os.makedirs(out_dir, exist_ok=True)

    # If DataFrame, pick columns as feature names
    if feature_names is None:
        if hasattr(X, "columns"):
            feature_names = list(X.columns)
        else:
            feature_names = [f"f{i+1}" for i in range(X.shape[1])]

    # Convert to numpy for sklearn
    if hasattr(X, "values"):
        X_np = X.values
    else:
        X_np = np.asarray(X)

    y_np = np.asarray(y).ravel()

    # Train/test split (stratified)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_np, y_np, test_size=test_size, random_state=random_state, stratify=y_np
        if len(np.unique(y_np)) > 1 else None
    )

    pipe = Pipeline([
        ("scaler", StandardScaler()),  # 標準化で収束性を改善
        ("clf", LinearSVC(
            C=C,
            tol=tol,
            max_iter=max_iter,
            dual=True,            # n_samples > n_features なら dual=False も可
            class_weight=class_weight,# 例: 'balanced'
            loss="squared_hinge",
            random_state=random_state
        ))
    ])

    # Fit
    pipe.fit(X_tr, y_tr)

    # Predictions & scores on test set
    y_pred = pipe.predict(X_te)
    # decision_function scores for ROC-AUC & hinge loss
    scores = pipe.decision_function(X_te)

    # Metrics
    acc = accuracy_score(y_te, y_pred)
    # roc_auc: use decision_function scores directly
    roc_auc = roc_auc_score(y_te, scores)

    # hinge_loss expects labels in {-1, +1}
    y_te_pm = np.where(y_te == 1, 1, -1)
    hloss = hinge_loss(y_te_pm, scores)

    # Save model
    #model_path = os.path.join(out_dir, model_filename)
    #joblib.dump(pipe, model_path)

    # Save metrics
    metrics = {
        "hinge_loss": float(hloss),
        "accuracy": float(acc),
        "roc_auc": float(roc_auc),
        "n_train": int(X_tr.shape[0]),
        "n_test": int(X_te.shape[0]),
        "features": int(X_np.shape[1]),
        "C": C,
        "class_weight": class_weight if class_weight is not None else "None",
        "random_state": random_state,
    }
    #metrics_path = os.path.join(out_dir, metrics_filename)
    #with open(metrics_path, "w", encoding="utf-8") as f:
    #    json.dump(metrics, f, ensure_ascii=False, indent=2)

    # Extract coefficients
    scaler = pipe.named_steps["scaler"]
    clf = pipe.named_steps["clf"]

    # LinearSVC coef_: shape (1, n_features) for binary
    coef_std = clf.coef_.ravel()  # coefficients in standardized space

    # Back to original feature scale: coef_original = coef_std / scaler.scale_
    if hasattr(scaler, "scale_") and scaler.scale_ is not None:
        coef_original = coef_std / scaler.scale_
    else:
        coef_original = coef_std.copy()

    #coef_df = pd.DataFrame(
    #    {
    #        "feature": feature_names,
    #        "coef_standardized": coef_std,
    #        "coef_original_scale": coef_original,
    #    }
    #).sort_values("coef_standardized", key=np.abs, ascending=False)

    #coefs_path = os.path.join(out_dir, coefs_filename)
    #coef_df.to_csv(coefs_path, index=False, encoding="utf-8")

    #print("Saved:")
    #print(f"  Model:   {model_path}")
    #print(f"  Metrics: {metrics_path}")
    #print(f"  Coefs:   {coefs_path}")

    return acc, hloss, roc_auc, coef_std, pipe
    
    #return {
    #    "model_path": model_path,
    #    "metrics_path": metrics_path,
    #    "coefs_path": coefs_path,
    #    "metrics": metrics,
    #}

# ===== サンプル実行 =====
#if __name__ == "__main__":
    # ダミーデータ（n_samples=1000, n_features=690）
#    rng = np.random.RandomState(0)
#    X_demo = rng.randn(1000, 690)
    # 非線形ではなく線形分離ぎみのターゲットを作る
#    true_w = rng.randn(690)
#    scores = X_demo @ true_w + 0.1 * rng.randn(1000)
#    y_demo = (scores > np.median(scores)).astype(int)

#    result = train_linear_svm_and_save(X_demo, y_demo)
#    print(json.dumps(result["metrics"], indent=2, ensure_ascii=False))


In [4]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, hinge_loss

def eval_on_validation(model, X_val, y_val):
    """
    Parameters
    ----------
    model : 学習済み分類器（LinearSVC/SVC/LogisticRegressionなど）
    X_val : 検証特徴量
    y_val : 検証ラベル（0/1 を想定）

    Returns
    -------
    dict : {"hinge_loss": ..., "accuracy": ..., "roc_auc": ...}
    """
    # 予測ラベル
    y_pred = model.predict(X_val)
    acc = accuracy_score(y_val, y_pred)

    # ROC-AUC 用スコア（優先: decision_function → 代替: predict_proba[:,1]）
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X_val)
    elif hasattr(model, "predict_proba"):
        scores = model.predict_proba(X_val)[:, 1]
    else:
        raise ValueError("ROC-AUC計算用のスコアが取得できません（decision_function / predict_proba が見つかりません）")

    roc_auc = roc_auc_score(y_val, scores)

    # hinge loss は y_true が {-1, +1} である必要がある
    y_val_pm1 = np.where(y_val == 1, 1, -1)
    hloss = hinge_loss(y_val_pm1, scores)

    return float(acc), float(hloss), float(roc_auc)
    #return {"hinge_loss": float(hloss), "accuracy": float(acc), "roc_auc": float(roc_auc)}

# 使い方例:
# metrics = eval_on_validation(trained_model, X_val, y_val)
# print(metrics)


In [5]:
import tqdm
import numpy as np

for n in tqdm.tqdm(range(5)):
    # 保存先ディレクトリを作成
    save_dir = os.path.join(os.path.join(dl_folder, f"model_{dl_number}"), f"model_{n}")
    #os.makedirs(save_dir, exist_ok=True)

    input_train = np.load(f"{save_dir}/{date}_{dl_number}_train_features.npy")
    trainY = np.load(f"{save_dir}/{date}_{dl_number}_train_targets.npy")
    input_valid = np.load(f"{save_dir}/{date}_{dl_number}_valid_features.npy")
    validY = np.load(f"{save_dir}/{date}_{dl_number}_valid_targets.npy")

    input_train = input_train.reshape(input_train.shape[0], input_train.shape[1]*input_train.shape[2])
    input_valid = input_valid.reshape(input_valid.shape[0], input_valid.shape[1]*input_valid.shape[2])

        # ファイル名にエポック番号を含めるフォーマットに変更
    model_file_path = os.path.join(
        save_dir,
        f"{date}_{dl_number}_{n}th_model.joblib"
    )

    accuracy, loss, auc, coef, model = train_linear_svm_and_save(input_train, trainY)
    val_acc, val_loss, val_auc = eval_on_validation(model, input_valid, validY)
    
    joblib.dump(model, model_file_path)
    np.save(os.path.join(save_dir, f"train_accuracy.npy"), accuracy)
    np.save(os.path.join(save_dir, f"valid_accuracy.npy"), val_acc)
    np.save(os.path.join(save_dir, f"train_loss.npy"), loss)
    np.save(os.path.join(save_dir, f"valid_loss.npy"), val_loss)
    np.save(os.path.join(save_dir, f"train_auc.npy"), auc)
    np.save(os.path.join(save_dir, f"valid_auc.npy"), val_auc)
    np.save(os.path.join(save_dir, f"coef.npy"), coef)

  0%|          | 0/5 [00:00<?, ?it/s]c:\Users\user\.conda\envs\deepshap\lib\site-packages\sklearn\svm\_base.py:986: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  "the number of iterations.", ConvergenceWarning)
 20%|██        | 1/5 [02:21<09:24, 141.06s/it]c:\Users\user\.conda\envs\deepshap\lib\site-packages\sklearn\svm\_base.py:986: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  "the number of iterations.", ConvergenceWarning)
 40%|████      | 2/5 [04:41<07:02, 140.74s/it]c:\Users\user\.conda\envs\deepshap\lib\site-packages\sklearn\svm\_base.py:986: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  "the number of iterations.", ConvergenceWarning)
 60%|██████    | 3/5 [07:03<04:42, 141.17s/it]c:\Users\user\.conda\envs\deepshap\lib\site-packages\sklearn\svm\_base.py:986: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  "the number of

In [7]:
import tqdm
import numpy as np

for n in tqdm.tqdm(range(5)):
    # 保存先ディレクトリを作成
    save_dir = os.path.join(os.path.join(dl_folder, f"model_{dl_number}"), f"model_{n}")
    #os.makedirs(save_dir, exist_ok=True)

    input_train = np.load(f"{save_dir}/{date}_{dl_number}_train_features.npy")
    trainY = np.load(f"{save_dir}/{date}_{dl_number}_train_targets.npy")
    input_valid = np.load(f"{save_dir}/{date}_{dl_number}_valid_features.npy")
    validY = np.load(f"{save_dir}/{date}_{dl_number}_valid_targets.npy")

    # ファイル名にエポック番号を含めるフォーマットに変更
    model_file_path = os.path.join(
        save_dir,
        f"{date}_{dl_number}_ver{ver}_epoch{{epoch:02d}}.h5"
    )

    # save_best_only=False にすることで、すべてのエポックを保存
    modelCheckpoint = ModelCheckpoint(
        filepath=model_file_path,
        monitor='val_loss',
        verbose=1,
        save_best_only=False,      # <- ここを False に
        save_weights_only=False,
        mode='min',
        save_freq='epoch'          # 毎エポック保存
    )

    output_dim = 32  # モデルの複雑さに応じて設定
    model = Sequential()
    model.add(GRU(output_dim,
                input_shape=(input_train.shape[1], input_train.shape[2]),
    #            kernel_regularizer=tf.keras.regularizers.L1L2(l1=1e-6, l2=1e-4)
                ))
    model.add(Dense(1, activation='sigmoid'))
    #model.summary()

    #model.compile(
    #    loss='mse',
    #    optimizer='adam',
    #    metrics=['mae']
    #)

    model.compile(
        loss=losses.BinaryCrossentropy(label_smoothing=0.001), 
        optimizer='adam', 
        metrics=['accuracy']
        )

    history = model.fit(
        x=input_train,
        y=trainY,
        validation_data=(input_valid, validY),
        epochs=10,
        batch_size=256,
        verbose=1,
        callbacks=[modelCheckpoint]
    )
    acc = history.history['acc']
    val_acc = history.history['val_acc']
    loss = history.history['loss']
    val_loss = history.history['val_loss']

    np.save(os.path.join(save_dir, f"train_accuracy.npy"), acc)
    np.save(os.path.join(save_dir, f"valid_accuracy.npy"), val_acc)
    np.save(os.path.join(save_dir, f"train_loss.npy"), loss)
    np.save(os.path.join(save_dir, f"valid_loss.npy"), val_loss)

  0%|          | 0/5 [00:00<?, ?it/s]

Train on 5800 samples, validate on 1350 samples
Epoch 1/10
5632/5800 [============================>.] - ETA: 0s - loss: 0.6843 - acc: 0.5712
Epoch 00001: saving model to D:/Python_TK_3/datas/251128_DL\model_Vis_angle_multi_01\model_0\251128_Vis_angle_multi_01_verGRU_epoch01.h5
5800/5800 [==============================] - 3s 520us/sample - loss: 0.6821 - acc: 0.5752 - val_loss: 0.6623 - val_acc: 0.6489
Epoch 2/10
5632/5800 [============================>.] - ETA: 0s - loss: 0.6459 - acc: 0.6637
Epoch 00002: saving model to D:/Python_TK_3/datas/251128_DL\model_Vis_angle_multi_01\model_0\251128_Vis_angle_multi_01_verGRU_epoch02.h5
5800/5800 [==============================] - 1s 249us/sample - loss: 0.6457 - acc: 0.6634 - val_loss: 0.6447 - val_acc: 0.6489
Epoch 3/10
5632/5800 [============================>.] - ETA: 0s - loss: 0.6367 - acc: 0.6662
Epoch 00003: saving model to D:/Python_TK_3/datas/251128_DL\model_Vis_angle_multi_01\model_0\251128_Vis_angle_multi_01_verGRU_epoch03.h5
5800/580

 20%|██        | 1/5 [00:18<01:12, 18.04s/it]

Train on 5750 samples, validate on 1400 samples
Epoch 1/10
5632/5750 [============================>.] - ETA: 0s - loss: 0.6690 - acc: 0.5977
Epoch 00001: saving model to D:/Python_TK_3/datas/251128_DL\model_Vis_angle_multi_01\model_1\251128_Vis_angle_multi_01_verGRU_epoch01.h5
5750/5750 [==============================] - 3s 523us/sample - loss: 0.6681 - acc: 0.5997 - val_loss: 0.6405 - val_acc: 0.6621
Epoch 2/10
5632/5750 [============================>.] - ETA: 0s - loss: 0.6379 - acc: 0.6612
Epoch 00002: saving model to D:/Python_TK_3/datas/251128_DL\model_Vis_angle_multi_01\model_1\251128_Vis_angle_multi_01_verGRU_epoch02.h5
5750/5750 [==============================] - 2s 279us/sample - loss: 0.6382 - acc: 0.6603 - val_loss: 0.6326 - val_acc: 0.6621
Epoch 3/10
5632/5750 [============================>.] - ETA: 0s - loss: 0.6274 - acc: 0.6609
Epoch 00003: saving model to D:/Python_TK_3/datas/251128_DL\model_Vis_angle_multi_01\model_1\251128_Vis_angle_multi_01_verGRU_epoch03.h5
5750/575

 40%|████      | 2/5 [00:36<00:54, 18.14s/it]

...

 60%|██████    | 3/5 [00:54<00:36, 18.37s/it]

Train on 5650 samples, validate on 1500 samples
Epoch 1/10
5632/5650 [============================>.] - ETA: 0s - loss: 0.6463 - acc: 0.6578
Epoch 00001: saving model to D:/Python_TK_3/datas/251128_DL\model_Vis_angle_multi_01\model_3\251128_Vis_angle_multi_01_verGRU_epoch01.h5
5650/5650 [==============================] - 3s 611us/sample - loss: 0.6460 - acc: 0.6582 - val_loss: 0.6356 - val_acc: 0.6733
Epoch 2/10
5632/5650 [============================>.] - ETA: 0s - loss: 0.6374 - acc: 0.6578
Epoch 00002: saving model to D:/Python_TK_3/datas/251128_DL\model_Vis_angle_multi_01\model_3\251128_Vis_angle_multi_01_verGRU_epoch02.h5
5650/5650 [==============================] - 2s 271us/sample - loss: 0.6375 - acc: 0.6577 - val_loss: 0.6247 - val_acc: 0.6733
Epoch 3/10
5632/5650 [============================>.] - ETA: 0s - loss: 0.6302 - acc: 0.6593
Epoch 00003: saving model to D:/Python_TK_3/datas/251128_DL\model_Vis_angle_multi_01\model_3\251128_Vis_angle_multi_01_verGRU_epoch03.h5
5650/565

 80%|████████  | 4/5 [01:14<00:18, 18.85s/it]

Train on 5650 samples, validate on 1500 samples
Epoch 1/10
5632/5650 [============================>.] - ETA: 0s - loss: 0.6409 - acc: 0.6610
Epoch 00001: saving model to D:/Python_TK_3/datas/251128_DL\model_Vis_angle_multi_01\model_4\251128_Vis_angle_multi_01_verGRU_epoch01.h5
5650/5650 [==============================] - 4s 682us/sample - loss: 0.6409 - acc: 0.6611 - val_loss: 0.6396 - val_acc: 0.6587
Epoch 2/10
5632/5650 [============================>.] - ETA: 0s - loss: 0.6269 - acc: 0.6623
Epoch 00002: saving model to D:/Python_TK_3/datas/251128_DL\model_Vis_angle_multi_01\model_4\251128_Vis_angle_multi_01_verGRU_epoch02.h5
5650/5650 [==============================] - 2s 306us/sample - loss: 0.6269 - acc: 0.6623 - val_loss: 0.6389 - val_acc: 0.6693
Epoch 3/10
5632/5650 [============================>.] - ETA: 0s - loss: 0.6125 - acc: 0.6681
Epoch 00003: saving model to D:/Python_TK_3/datas/251128_DL\model_Vis_angle_multi_01\model_4\251128_Vis_angle_multi_01_verGRU_epoch03.h5
5650/565

100%|██████████| 5/5 [01:34<00:00, 18.85s/it]
